# Day 7 Lab 02: Spark SQL Exploration

        **Layer:** Spark fundamentals  
        **Python reference:** `Lab_Files/labs/lab_02_spark_sql_exploration.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Register raw order events as a temp view.
- Run SQL aggregations in Spark.
- Write and inspect status-level event counts.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [ ]:
from pathlib import Path
import os
import sys

HERE = Path.cwd().resolve()
LAB_FILES = HERE / "Lab_Files"
if not LAB_FILES.exists():
    LAB_FILES = HERE.parent / "Lab_Files"

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")

## 1. Import Lab Helpers

In [ ]:
from day7_common import OUTPUT_DIR, ORDER_SOURCE_FILES, ensure_output_dirs, read_order_events, require_source_data, spark_session, write_csv_dir

## 2. Start Spark

In [ ]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook02SparkSQL")

## 3. Read Batch 1 Raw Events

In [ ]:
orders = read_order_events(spark, [ORDER_SOURCE_FILES[0]])
orders.printSchema()
orders.show(10, truncate=False)

## 4. Register a Temp View

In [ ]:
orders.createOrReplaceTempView("raw_orders")

## 5. Run Spark SQL

In [ ]:
status_counts = spark.sql(
    '''
    SELECT
      status,
      COUNT(*) AS event_count,
      ROUND(SUM(amount), 2) AS raw_amount_total
    FROM raw_orders
    GROUP BY status
    ORDER BY event_count DESC, status
    '''
)

## 6. Inspect and Write Output

In [ ]:
status_counts.show(truncate=False)
write_csv_dir(status_counts, OUTPUT_DIR / "notebook_02_status_counts")
print(f"Raw events analyzed from batch 1: {orders.count()}")

## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [ ]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()